# Team 11 Project: DreamerV3 Implementation and Extension

**Course**: Deep Reinforcement Learning (AI.61100_2026_1)  
**Base Paper**: [*Mastering Diverse Domains through World Models (DreamerV3)*](https://arxiv.org/abs/2301.04104)  
**Reference Papers**: 
* [*World Models*](https://arxiv.org/abs/1803.10122)
* [*Learning Latent Dynamics for Planning from Pixels*](https://arxiv.org/abs/1811.04551)
* [*Mastering Atari with Discrete World Models*](https://arxiv.org/abs/2010.02193)

---

### Team 11 Members
* Hyeonseo Yun
* Kihyun Seol
* Seungmin Cha
* Seungyeon Ryu

### Reproducible local setup (GitHub/Hugging Face)

Run these steps in a **fresh local clone** (no `/mnt/...` assumptions):

1. Clone branch `dreamerv2-v3` and create Python 3.11 environment.
2. Install deps: `pip install -U -r requirements.txt huggingface_hub gymnasium highway-env imageio ruamel.yaml`.
3. (Optional) set HF token: `export HF_TOKEN=...` for private/quota-safe downloads.
4. Open this notebook at repo root and run cells top-to-bottom.
5. Default mode uses cached artifacts when present; set env flags only when needed:
   - `RUN_MC_LIVE=1` for live Minecraft rollout
   - `REFRESH_ATARI_GIFS=1` for 3-game Atari GIF refresh
   - `RUN_ATARI_LIVE=1` for timed retraining pipeline
6. Highway checkpoint is fetched from HF repo `HyunseoYun/dreamerv3-custom-envs` automatically when missing.

Expected outputs:
- `report/REPORT.md`, `report/ANALYSIS.md`
- `report/atari_compare/ATARI_COMPARE.md`
- `highlights/inference/*.gif` and `highway_roundabout_inference.gif`


In [ ]:
import io, os, pathlib, sys, urllib.request, zipfile
from IPython.display import Image, display, Markdown
%matplotlib inline

REPO, BRANCH = 'franktome/Dreamerv3_RL_project', 'dreamerv2-v3'
WORKSPACE = pathlib.Path('.').resolve()
if pathlib.Path('dvbench/__init__.py').exists():
    sys.path.insert(0, str(WORKSPACE))
else:
    cache = pathlib.Path.home() / '.cache' / 'dvbench_pkg'
    cache.mkdir(parents=True, exist_ok=True)
    url = f'https://github.com/{REPO}/archive/refs/heads/{BRANCH}.zip'
    with urllib.request.urlopen(url, timeout=180) as r:
        data = r.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        zf.extractall(cache)
    sys.path.insert(0, str(next(cache.glob(f'*-{BRANCH}'))))

from dvbench.paths import default_paths
from dvbench.env_setup import clone_dreamerv3, setup_jax, ensure_xvfb
from dvbench.hf_assets import login_if_needed, resolve_minecraft_logdir
from dvbench import inference_demo
from dvbench import viz_advanced

login_if_needed()
cfg = default_paths(WORKSPACE, gpu='1')
clone_dreamerv3(cfg)
setup_jax(cfg)
MC_LOGDIR = resolve_minecraft_logdir(cfg)
print('Minecraft logdir:', MC_LOGDIR)


## 1.1 — Benchmark Results: Minecraft & Atari 57

Official published scores for Minecraft (V3 / PPO / IMPALA) and Atari 57 (V2 vs V3).
Local Minecraft training episodes overlaid when available.

In [ ]:
adv = viz_advanced.run_advanced_viz(cfg, logdir=MC_LOGDIR, show=False)
for key in ['minecraft_enhanced', 'minecraft_overlay', 'atari_heatmap', 'atari_10_curves', 'atari_10_panels']:
    p = adv.get(key, '')
    if p and pathlib.Path(p).exists():
        display(Image(filename=p))
display(Markdown(
    f"**Atari median HNS:** V2={adv['median_hns_v2']:.2f} → V3={adv['median_hns_v3']:.2f} "
    f"({adv['v3_win_pct']:.0f}% games favor V3)"
))
display(Markdown(f"**10-game set:** {', '.join(adv['games'])}"))


### ✅ Analysis

- **Minecraft:** V3 reaches deeper milestones faster than PPO/IMPALA on official runs. PPO/IMPALA improve over time but lag on task-completion rate.
- **Atari:** V3 median HNS exceeds V2; largest gains appear on sparse-reward games (e.g. demon_attack, asterix).
- **Local overlay:** scattered points show our ongoing training run; rolling mean tracks recent episode quality.

## 1.2 — Minecraft env rollout (DreamerV3)

Policy rollout from the local checkpoint. Milestone strip captures frames when inventory progress advances.

In [ ]:
from dvbench.inference_demo import analyze_training_health

mc_gif = cfg.highlights_dir / 'inference' / 'minecraft_diamond_rollout.gif'
mc_strip = cfg.highlights_dir / 'inference' / 'minecraft_milestone_strip.gif'
USE_CACHED = mc_gif.exists() and not os.environ.get('RUN_MC_LIVE')

if USE_CACHED:
    health = analyze_training_health(MC_LOGDIR)
    mc_result = {
        'ok': True,
        'gif': str(mc_gif),
        'strip_gif': str(mc_strip) if mc_strip.exists() else '',
        'reward': health.get('max_episode_score', 0),
        'max_milestone': health.get('max_milestone_name', '?'),
        'milestones_reached': [],
        'steps': health.get('step', '?'),
        'mean_episode_score': health.get('mean_episode_score', 0),
        'training_episodes': health.get('episodes', 0),
    }
elif (MC_LOGDIR / 'ckpt').exists():
    ensure_xvfb(cfg.display)
    mc_result = inference_demo.run_minecraft_env_gif(cfg, logdir=MC_LOGDIR, max_steps=400)
else:
    mc_result = {'ok': False, 'error': 'No checkpoint'}

if mc_result.get('ok'):
    display(Markdown(
        f"**Rollout** — max episode reward={mc_result.get('reward', 0):.1f}, "
        f"mean episode score={mc_result.get('mean_episode_score', 0):.2f}, "
        f"max milestone={mc_result.get('max_milestone', '?')}, "
        f"training step={mc_result.get('steps', '?')}"
    ))
    if mc_result.get('milestones_reached'):
        display(Markdown('**Progress:** ' + ' → '.join(mc_result['milestones_reached'])))
    if pathlib.Path(mc_result.get('gif', '')).exists():
        display(Image(filename=mc_result['gif']))
    if mc_result.get('strip_gif') and pathlib.Path(mc_result['strip_gif']).exists():
        display(Image(filename=mc_result['strip_gif']))
else:
    display(Markdown(f"*{mc_result.get('error', 'skipped')}*"))


### ✅ Analysis

- Episode **reward** reflects cumulative milestone index during the rollout.
- **Milestone strip** labels each inventory unlock; wider strip = more progress within the episode window.
- Compare with A1 official curves to see where this checkpoint sits relative to published V3 runs.

## 1.3 — Atari Key Game Tasks Comparison

Ten games: eight largest V3 gains plus two where V2 remains competitive. Score trajectories from official 50M-step runs.

In [ ]:
display(Image(filename=str(cfg.report_dir / 'atari_10game_bars.png')))
display(Image(filename=str(cfg.report_dir / 'atari_v2_v3.png')))
top = adv['compare'].sort_values('delta', ascending=False).head(5)[['DreamerV2', 'DreamerV3', 'delta']]
top.index = [i.replace('atari_', '') for i in top.index]
top


### ✅ Analysis

- **Left panels (V2) vs right panels (V3)** in the 10-game figure show learning speed and asymptotic HNS per title.
- Bar chart highlights per-game gaps; positive delta means V3 exceeds V2 at 50M steps.
- Live env GIFs require per-game checkpoints; this notebook uses official score trajectories for reproducibility.

## 1.4 Executive summary

In [ ]:
from dvbench import report
display(Image(filename=str(cfg.report_dir / 'executive_summary.png')))
report.write_report(cfg, adv['mc_summary'], adv['compare'], local_mc=mc_result if mc_result.get('ok') else None, advanced=adv)
analysis = (cfg.report_dir / 'ANALYSIS.md').read_text(encoding='utf-8')
display(Markdown(analysis))
display(Markdown(f"Artifacts: `{cfg.report_dir / 'REPORT.md'}`"))


### ✅ Analysis

- **Atari:** V3 improves median HNS and wins most per-game comparisons.
- **Minecraft:** V3 shows highest mean milestone depth among compared baselines on official data.
- **Local run:** supplements official curves with checkpoint-specific rollout behavior.

---

# 2. Atari Benchmark Evaluations: DreamerV2 vs DreamerV3 (3 Key Games)

Games: **pong**, **breakout**, **boxing** — inference (V2 left | V3 right) and environment perturbation evaluation.

| Mode | When to use |
|------|-------------|
| **Default (cached)** | Show existing GIFs under `highlights/inference/` |
| `REFRESH_ATARI_GIFS=1` | Re-run inference |
| `RUN_ATARI_LIVE=1` | Full 90 min/model retrain + infer |


In [ ]:
# B0 — setup
from pathlib import Path
from IPython.display import Image, display, Markdown
import os

from dvbench.paths import default_paths
from dvbench import atari_compare, atari_anim
from dvbench.atari_anim import AnimSpec, ANIM_PRESETS

cfg = default_paths(Path('.').resolve(), gpu='1')
cfg.apply_env(mem_fraction=0.25)
GAMES = atari_compare.GAMES
display(Markdown(f"**Games:** {', '.join(GAMES)}"))


## 2.1 — Training & inference pipeline

Runs smoke-tested timed training (V2 TF + V3 JAX sequential) then compare GIFs.

In [ ]:
RUN_LIVE = bool(os.environ.get('RUN_ATARI_LIVE'))
REFRESH = bool(os.environ.get('REFRESH_ATARI_GIFS'))

if RUN_LIVE:
    smoke = atari_compare.run_smoke(cfg)
    display(Markdown(f"Smoke: **{'OK' if smoke['ok'] else 'FAILED'}**"))
    if smoke['ok']:
        atari_compare.run_timed_training(cfg, minutes_per_run=90)
        pipeline = atari_compare.run_full_pipeline(cfg, smoke=False, skip_train=True)
elif REFRESH:
    display(Markdown('**Refreshing** compare + anim GIFs from current checkpoints (no retrain)…'))
    from dvbench import gif_compare, viz_atari_compare
    refresh = {'compare': {}, 'anim': {}}
    for game in GAMES:
        refresh['compare'][game] = gif_compare.infer_both(cfg, game, max_steps=1500)
        refresh['anim'][game] = {}
        for preset in ('fast', 'sluggish'):
            refresh['anim'][game][preset] = atari_anim.run_anim_compare(
                cfg, game, preset=preset, max_steps=1200)
    metrics = atari_compare.collect_metrics(cfg)
    viz_atari_compare.generate_all(cfg, metrics, GAMES)
    viz_atari_compare.write_atari_compare_report(cfg, metrics, refresh)
    pipeline = {'refreshed': True, 'games': list(GAMES)}
    display(Markdown('Refresh complete.'))
else:
    cache = cfg.report_dir / 'atari_compare' / 'pipeline_result.json'
    pipeline = {'cached': True, 'report': str(cfg.report_dir / 'atari_compare' / 'ATARI_COMPARE.md')}
    if cache.exists():
        display(Markdown(f"Using cached pipeline metadata: `{cache}`"))
    else:
        display(Markdown(
            'Using cached GIFs/plots on disk. Set `REFRESH_ATARI_GIFS=1` after long training '
            'to regenerate side-by-side GIFs from latest checkpoints.'
        ))

display(Markdown(f"Report: `{cfg.report_dir / 'atari_compare' / 'ATARI_COMPARE.md'}`"))


## 2.1b — Fair Atari Retrain (100k env steps)

Controlled retrain of **pong / breakout / boxing** with identical settings (`atari atari_compare`, repeat=4, sticky=0.25, noops=30, grayscale). Budget: **100,000 environment steps** per run (~400k logged frames). Comparison uses `aligned_env_steps = min(V2, V3)`; GIFs load V2 sidecar snapshots and V3 checkpoints at or before the aligned step.

Checkpoints: [Hugging Face `checkpoints/atari_*`](https://huggingface.co/HyunseoYun/dreamerv3-custom-envs/tree/main/checkpoints)

In [ ]:
# Fair retrain summary (generated after controlled 100k-step training)
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

from dvbench import atari_align, atari_compare

fair_json = cfg.report_dir / 'atari_compare' / 'fair_retrain_result.json'
if fair_json.exists():
    fair = json.loads(fair_json.read_text())
    align_df = pd.DataFrame(fair['alignment'])
else:
    align_df = pd.DataFrame(atari_align.alignment_table(cfg, GAMES))

display(Markdown('### Alignment (fair cutoff per game)'))
show = align_df[['game', 'v2_env_steps', 'v2_episodes', 'v3_env_steps', 'v3_episodes',
                 'aligned_env_steps', 'aligned_episodes', 'fair_gif']].copy()
show.columns = ['game', 'V2 steps', 'V2 ep', 'V3 steps', 'V3 ep', 'aligned steps', 'aligned ep', 'fair GIF']
display(show)

metrics = atari_compare.collect_metrics(cfg)
display(Markdown('### Metrics at aligned steps'))
display(metrics['summary'])

lc = cfg.report_dir / 'atari_compare' / 'learning_curves_3games.png'
if lc.exists():
    display(Markdown('### Learning curves (truncated to aligned steps)'))
    display(Image(filename=str(lc)))

display(Markdown('### Side-by-side GIFs (aligned checkpoints)'))
for game in GAMES:
    gif = cfg.highlights_dir / 'inference' / f'atari_{game}_v2v3_compare.gif'
    row = align_df[align_df['game'] == game].iloc[0]
    display(Markdown(
        f"**{game}** — aligned **{int(row['aligned_env_steps']):,}** env steps "
        f"(V2 ep {int(row['v2_episodes'])}, V3 ep {int(row['v3_episodes'])}) "
        f"{'✅ fair GIF' if row['fair_gif'] else '⚠️ check alignment'}"
    ))
    if gif.exists():
        display(Image(filename=str(gif)))

display(Markdown(f"Full report: `{cfg.report_dir / 'atari_compare' / 'ATARI_COMPARE.md'}`"))
display(Markdown(f"Alignment details: `{cfg.report_dir / 'atari_compare' / 'ALIGNMENT.md'}`"))

## 2.2 — Comparison Between DreamerV2 & DreamerV3

In [ ]:
from datetime import datetime

def _mtime(p):
    return datetime.fromtimestamp(p.stat().st_mtime).strftime('%Y-%m-%d %H:%M') if p.exists() else 'missing'

for game in GAMES:
    gif = cfg.highlights_dir / 'inference' / f'atari_{game}_v2v3_compare.gif'
    v2_ckpt = cfg.atari_logdir(game, 'v2') / 'variables.pkl'
    v3_ckpt = cfg.atari_logdir(game, 'v3') / 'ckpt' / 'latest'
    stale = gif.exists() and v2_ckpt.exists() and gif.stat().st_mtime < v2_ckpt.stat().st_mtime
    if gif.exists():
        note = ' *(GIF older than v2 ckpt — run B1 with REFRESH_ATARI_GIFS=1)*' if stale else ''
        display(Markdown(
            f"### {game} — V2 | V3  (gif {_mtime(gif)}, ckpt v2 {_mtime(v2_ckpt)}, v3 {_mtime(v3_ckpt)}){note}"
        ))
        display(Image(filename=str(gif)))
    else:
        display(Markdown(
            f"*{game}: no compare GIF — run B1 with REFRESH_ATARI_GIFS=1 (or RUN_ATARI_LIVE=1)*"
        ))


## 2.3 Environment Perturbations Evaluation

Each GIF stacks **baseline V2|V3** (top) and **perturbed V2|V3** (bottom).

- **Cached presets** below: `fast`, `sluggish`
- **Interactive:** use **B3b** sliders + **Run anim compare** button (repeat / sticky / noops)

In [ ]:
# Edit perturbation below, then set RUN_CUSTOM_ANIM=1 to regenerate 
custom = AnimSpec(repeat=2, sticky=0.25, noops=30)
display(Markdown(
    f"**Presets:** {', '.join(ANIM_PRESETS)} | custom repeat={custom.repeat} sticky={custom.sticky}"
))

if os.environ.get('RUN_CUSTOM_ANIM'):
    demo_game = GAMES[0]
    custom_result = atari_anim.run_anim_compare(
        cfg, demo_game, preset='custom',
        baseline=ANIM_PRESETS['baseline'], perturbed=custom, max_steps=800)
    display(Markdown(f"Custom anim ({demo_game}): `{custom_result.get('grid_gif', '')}`"))
    if custom_result.get('grid_gif') and Path(custom_result['grid_gif']).exists():
        display(Image(filename=custom_result['grid_gif']))

for game in GAMES:
    for preset in ('fast', 'sluggish'):
        gif = cfg.highlights_dir / 'inference' / 'anim' / f'atari_{game}_{preset}_v2v3.gif'
        if gif.exists():
            display(Markdown(f"**{game} — {preset}**"))
            display(Image(filename=str(gif)))
        else:
            display(Markdown(f"*{game} — {preset}: missing (run B1 with REFRESH_ATARI_GIFS=1)*"))


### ✅ Analysis

- **fast** (`repeat=2`): shorter frame skip → faster ball/paddle dynamics.
- **sluggish** (`sticky=0.5`): actions repeat more often → delayed response.
- Compare V2 vs V3 sensitivity to the same perturbation in the GIF grids.

----

# 3. Highway Environment Inference & Evaluation

In [ ]:
# pyvirtualdisplay 없이 바로 렌더링
import gymnasium as gym
import highway_env

env = gym.make('roundabout-v0', render_mode='rgb_array')
print(env.observation_space.shape)
print(env.action_space.n)
obs, _ = env.reset()
frame = env.render()  # 화면 없이 numpy array로 반환
print(frame.shape)    # (H, W, 3) 이면 성공

In [ ]:
# 셀 1 - pyvirtualdisplay 제거, 환경변수만 설정
# JAX/XLA는 GPU 커널 컴파일 시 /tmp에 PTX를 씁니다. 루트 디스크가 가득 차면
# Agent 초기화에서 RESOURCE_EXHAUSTED가 납니다. 반드시 jax import 전에 실행하세요.
import sys
import os
import importlib
import pathlib as _pl

WORKSPACE = _pl.Path('.').resolve()
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

# A0 등 이전 셀에서 seollab을 import했으면 구버전이 캐시될 수 있음
for _mod in ('seollab.paths', 'seollab'):
    if _mod in sys.modules:
        importlib.reload(sys.modules[_mod])

from dvbench.paths import default_paths
cfg = default_paths(WORKSPACE, gpu='0')
cfg.apply_env()  # TMPDIR / JAX cache 설정 포함
print('TMPDIR:', os.environ['TMPDIR'])

import jax
print('JAX devices:', jax.devices())


In [ ]:
import elements

# 기존 YAML 읽기 부분을 대체하는 하드코딩된 파이썬 딕셔너리
config_dict = {
    "loss_scales": {
        "rec": 1.0, "rew": 1.0, "con": 1.0, "dyn": 1.0,
        "rep": 0.1, "policy": 1.0, "value": 1.0, "repval": 0.3
    },
    "opt": {
        "lr": 4e-05, "agc": 0.3, "eps": 1e-20, "beta1": 0.9,
        "beta2": 0.999, "momentum": True, "wd": 0.0,
        "schedule": "const", "warmup": 1000, "anneal": 0
    },
    "ac_grads": False,
    "dyn": {
        "typ": "rssm",
        "rssm": {
            "deter": 2048, "hidden": 256, "stoch": 32, "classes": 16,
            "act": "silu", "norm": "rms", "unimix": 0.01,
            "outscale": 1.0, "winit": "trunc_normal_in",
            "imglayers": 2, "obslayers": 1, "dynlayers": 1,
            "absolute": False, "blocks": 8, "free_nats": 1.0
        }
    },
    "enc": {
        "typ": "simple",
        "simple": {
            "depth": 16, "mults": [2, 3, 4, 4], "layers": 3,
            "units": 256, "act": "silu", "norm": "rms",
            "winit": "trunc_normal_in", "symlog": True,
            "outer": False, "kernel": 5, "strided": False
        }
    },
    "dec": {
        "typ": "simple",
        "simple": {
            "depth": 16, "mults": [2, 3, 4, 4], "layers": 3,
            "units": 256, "act": "silu", "norm": "rms",
            "outscale": 1.0, "winit": "trunc_normal_in",
            "outer": False, "kernel": 5, "bspace": 8, "strided": False
        }
    },
    "rewhead": {
        "layers": 1, "units": 256, "act": "silu", "norm": "rms",
        "output": "symexp_twohot", "outscale": 0.0,
        "winit": "trunc_normal_in", "bins": 255
    },
    "conhead": {
        "layers": 1, "units": 256, "act": "silu", "norm": "rms",
        "output": "binary", "outscale": 1.0, "winit": "trunc_normal_in"
    },
    "policy": {
        "layers": 3, "units": 256, "act": "silu", "norm": "rms",
        "minstd": 0.1, "maxstd": 1.0, "outscale": 0.01,
        "unimix": 0.01, "winit": "trunc_normal_in"
    },
    "value": {
        "layers": 3, "units": 256, "act": "silu", "norm": "rms",
        "output": "symexp_twohot", "outscale": 0.0,
        "winit": "trunc_normal_in", "bins": 255
    },
    "policy_dist_disc": "categorical",
    "policy_dist_cont": "bounded_normal",
    "imag_last": 0,
    "imag_length": 15,
    "horizon": 333,
    "contdisc": True,
    "imag_loss": {"slowtar": False, "lam": 0.95, "actent": 0.0003, "slowreg": 1.0},
    "repl_loss": {"slowtar": False, "lam": 0.95, "slowreg": 1.0},
    "slowvalue": {"rate": 0.02, "every": 1},
    "retnorm": {"impl": "perc", "rate": 0.01, "limit": 1.0, "perclo": 5.0, "perchi": 95.0, "debias": False},
    "valnorm": {"impl": "none", "rate": 0.01, "limit": 1e-08},
    "advnorm": {"impl": "none", "rate": 0.01, "limit": 1e-08},
    "reward_grad": True,
    "repval_loss": True,
    "repval_grad": True,
    "report": True,
    "report_gradnorms": False,
    "logdir": "logdir/highway_roundabout",
    "seed": 0,
    "jax": {
        "platform": "cuda", "compute_dtype": "bfloat16",
        "policy_devices": [0], "train_devices": [0],
        "mock_devices": 0, "prealloc": True, "jit": True,
        "debug": False, "expect_devices": 0, "enable_policy": True,
        "coordinator_address": ""
    },
    "batch_size": 16,
    "batch_length": 64,
    "replay_context": 1,
    "report_length": 32,
    "replica": 0,
    "replicas": 1
}

config =elements.Config(
    config_dict
)

In [ ]:
# 셀 2 - 환경 + 에이전트 로드
import os
import sys
import importlib
import pathlib

if 'seollab.paths' in sys.modules:
    importlib.reload(sys.modules['seollab.paths'])
from dvbench.paths import default_paths
default_paths('.').apply_env()
print('TMPDIR:', os.environ['TMPDIR'])

import gymnasium as gym
import highway_env
import numpy as np
import imageio
import elements
import ruamel.yaml as yaml
from dreamerv3.agent import Agent
from dvbench.hf_assets import ensure_checkpoints

DOWNLOAD_DIR = ensure_checkpoints(pathlib.Path('.'))

env = gym.make('roundabout-v0', render_mode='rgb_array')

obs_space = {
    "obs": elements.Space(np.float32, env.observation_space.shape),
    "reward": elements.Space(np.float32),
    "is_first": elements.Space(bool),
    "is_last": elements.Space(bool),
    "is_terminal": elements.Space(bool),
}
act_space = {"action": elements.Space(np.int32, (), 0, env.action_space.n)}

agent = Agent(obs_space, act_space, config)
cp = elements.Checkpoint(pathlib.Path(DOWNLOAD_DIR) / 'checkpoints/highway_roundabout')
cp.agent = agent
print('ckpt:', list((pathlib.Path(DOWNLOAD_DIR) / 'checkpoints/highway_roundabout').iterdir()))
cp.load()
print('체크포인트 로드 완료!')


In [ ]:
# 셀 3 - 추론 + GIF 저장
env = gym.make('roundabout-v0', render_mode='rgb_array', config={'duration': 100})
frames = []
obs_raw, _ = env.reset()
done = False
total_reward = 0
carry = agent.init_policy(1)
is_first = True

while not done:
    frame = env.render()
    frames.append(frame)

    obs = {
        "obs": np.array([obs_raw], dtype=np.float32),        # (1, 5, 5)
        "reward": np.array([0.0], dtype=np.float32),          # (1,)
        "is_first": np.array([is_first]),                     # (1,)
        "is_last": np.array([False]),                         # (1,)
        "is_terminal": np.array([False]),                     # (1,)
    }
    is_first = False

    carry, act, _ = agent.policy(carry, obs, mode='eval')
    action = int(act['action'][0])
    obs_raw, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    done = terminated or truncated

env.close()
print(f"총 점수: {total_reward:.2f}, 프레임 수: {len(frames)}")

imageio.mimsave('highway_roundabout_inference.gif', frames, fps=10)
print("GIF 저장 완료!")

In [ ]:
# 셀 4 - 노트북에서 바로 보기
from IPython.display import Image
Image('highway_roundabout_inference.gif')